In [2]:
import jax.numpy as jnp
import jax.random as jr
import numpyro
import numpyro.distributions as dist
from numpyro.infer import Predictive

import dynestyx as dsx
from dynestyx import flatten_draws
from dynestyx import (
    ContinuousTimeStateEvolution,
    DynamicalModel,
    LinearGaussianObservation,
    SDESimulator,
)

state_dim = 3
observation_dim = 1


def l63_model(obs_times=None, obs_values=None, predict_times=None):
    rho = numpyro.sample("rho", dist.Uniform(10.0, 40.0))
    dynamics = DynamicalModel(
        initial_condition=dist.MultivariateNormal(
            loc=jnp.zeros(state_dim), covariance_matrix=20.0**2 * jnp.eye(state_dim)
        ),
        state_evolution=ContinuousTimeStateEvolution(
            drift=lambda x, u, t: jnp.array(
                [
                    10.0 * (x[1] - x[0]),
                    x[0] * (rho - x[2]) - x[1],
                    x[0] * x[1] - (8.0 / 3.0) * x[2],
                ]
            ),
            diffusion_coefficient=lambda x, u, t: jnp.eye(3),
        ),
        observation_model=LinearGaussianObservation(
            H=jnp.eye(observation_dim, state_dim),  # observe only x[0]
            R=jnp.eye(observation_dim),
        ),
    )
    return dsx.sample("f", dynamics, obs_times=obs_times, obs_values=obs_values, predict_times=predict_times)


Generate data below

In [ ]:
key = jr.PRNGKey(0)
rho_true = 28.0
T_forecast = 8.0
# Generate longer trajectory: training window + held-out future for rollout evaluation
obs_times_full = jnp.arange(0.0, 20.0 + T_forecast, 0.01)  # 0..28

predictive = Predictive(
    l63_model,
    params={"rho": jnp.array(rho_true)},
    num_samples=1,
    exclude_deterministic=False,
)
with SDESimulator(source="em_scan"):
    synthetic = predictive(key, predict_times=obs_times_full)

# Simulators return (n_sim, T, dim); Predictive adds (num_samples,).
# With num_samples=1 and n_sim=1 we index explicitly.
print(
    "synthetic shapes:",
    synthetic["f_times"].shape,
    synthetic["f_states"].shape,
    synthetic["f_observations"].shape,
)
times = synthetic["f_times"][0, 0, :]
states = synthetic["f_states"][0, 0, :, :]  # (T, 3)
observations = synthetic["f_observations"][0, 0, :, :]  # (T, 1)

# Training portion (0..20) for MCMC; future (20..28) withheld for rollout eval
mask_train_full = times <= 20.0
times_train_full = times[mask_train_full]
observations_train = observations[mask_train_full]

times_test_full = times[~mask_train_full]
observations_test_full = observations[~mask_train_full]

Plotting generated data

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(times, states[:, 0], label="x1")
axes[0].plot(times, states[:, 1], label="x2")
axes[0].plot(times, states[:, 2], label="x3")
axes[0].set_ylabel("state")
axes[0].legend(loc="upper right")
axes[1].plot(
    times, observations[:, 0], label="obs (x1 + noise)", color="C0", alpha=0.8
)
axes[1].set_ylabel("observation")
axes[1].set_xlabel("time")
axes[1].legend()
plt.tight_layout()
plt.show()

Evaluating the rollout quality: largely the same logic as before

In [ ]:
rho_post_mean = jnp.mean(posterior["rho"])
n_sim = 30
#Number of parameter candidates we pick from the posterior
num_samples = 2  # Change this to 1 or >1 to test both cases
#Difference with only having 1 sample: 
# - More variability in predictions
# slowdown due to more simulations (n_sim * num_samples)

predictive = Predictive(
    l63_model,
    params={"rho": jnp.array(rho_post_mean)},
    num_samples=num_samples,
    exclude_deterministic=False,
)
with SDESimulator(n_simulations=n_sim, source="em_scan"):
    with Filter(filter_config=ContinuousTimeEnKFConfig(n_particles=50, record_filtered_states_mean=True, record_filtered_states_cov_diag=True)):
        samples = predictive(
            jr.PRNGKey(99),
            obs_times=times_train_full,
            obs_values=observations_train,
            predict_times=times_test_full,
        )

pred_states = jnp.asarray(samples["f_predicted_states"])  # (num_samples, n_sim, T_pred, 3)
pred_times_arr = jnp.asarray(samples["f_predicted_times"])  # (num_samples, n_sim, T_pred)
filtered_means = jnp.asarray(samples["f_filtered_states_mean"])  # (num_samples, T_train, 3)
filtered_cov_diag = jnp.asarray(samples["f_filtered_states_cov_diag"])  # (num_samples, T_train, 3)

print(
    "rollout shapes:",
    pred_states.shape,
    pred_times_arr.shape,
    filtered_means.shape,
    filtered_cov_diag.shape,
)

# flatten_draws merges (num_samples, n_sim, T, D) → (num_samples*n_sim, T, D).
# Filter outputs (f_filtered_states_mean, f_filtered_states_cov_diag) are (num_samples, T, D) — no n_sim.
pred_draws = flatten_draws(pred_states)
pred_t = flatten_draws(pred_times_arr)[0]
filtered_mean_med = jnp.percentile(filtered_means, 50.0, axis=0)
filtered_std_med = jnp.sqrt(jnp.percentile(filtered_cov_diag, 50.0, axis=0))

# Plot: true states, filtered means, observations, predicted CI
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
lo = jnp.percentile(pred_draws, 2.5, axis=0)
hi = jnp.percentile(pred_draws, 97.5, axis=0)
state_labels = [r"x1", r"x2", r"x3"]
for i, ax in enumerate(axes):
    ax.fill_between(pred_t, lo[:, i], hi[:, i], alpha=0.3, label="95% CI (rollout)")
    ax.fill_between(
        times_train_full,
        filtered_mean_med[:, i] - 2 * filtered_std_med[:, i],
        filtered_mean_med[:, i] + 2 * filtered_std_med[:, i],
        alpha=0.25,
        color="green",
        label="Filtered ±2σ",
    )
    ax.plot(times_train_full, states[mask_train_full][:, i], "k--", label="True (train)", lw=1)
    ax.plot(times_test_full, states[~mask_train_full][:, i], "k:", lw=1.5, label="True (future, held-out)")
    ax.plot(times_train_full, filtered_mean_med[:, i], "g.-", markersize=4, label="Filtered mean")
    if i == 0:
        ax.scatter(times_train_full, observations[mask_train_full][:, 0], color="C3", marker="x", s=30, label="Observed")
        # ax.scatter(times_test_full, observations[~mask_train_full][:, 0], color="C4", marker="+", s=30, label="Future (held-out)")
    ax.set_ylabel(state_labels[i])
    ax.legend(loc="upper right", fontsize=8)
    ax.axvline(times_train_full[-1], color="gray", linestyle=":", alpha=0.7)
axes[0].set_title("Filter + SDESimulator: rollout with predict_times")
axes[-1].set_xlabel("time")
plt.tight_layout()
plt.show()

Algorithm for Backtesting

In [ ]:
# Generate a NEW dataset for back-testing (different trajectory)
key_backtest = jr.PRNGKey(123)
times_backtest = jnp.arange(0.0, 15.0, 0.01)  # shorter window for demo
with SDESimulator(source="em_scan"):
    synthetic_backtest = Predictive(
        l63_model,
        params={"rho": jnp.array(rho_true)},
        num_samples=1,
        exclude_deterministic=False,
    )(key_backtest, predict_times=times_backtest)
print(
    "backtest synthetic shapes:",
    synthetic_backtest["f_times"].shape,
    synthetic_backtest["f_states"].shape,
    synthetic_backtest["f_observations"].shape,
)
states_backtest = synthetic_backtest["f_states"][0, 0, :, :]
observations_backtest = synthetic_backtest["f_observations"][0, 0, :, :]
times_backtest = synthetic_backtest["f_times"][0, 0, :]

# Sparse observations: every 50th timepoint (as would arrive in real time)
downsample = 50
obs_times_sparse = times_backtest[::downsample]
obs_values_sparse = observations_backtest[::downsample]

# Dense predict_times: full grid (interpolation between sparse obs)
predict_times_dense = times_backtest

# Apply learned model: posterior-predictive with Filter + SDESimulator
predictive_interp = Predictive(
    l63_model,
    params={"rho": jnp.array(rho_post_mean)},
    num_samples=3,
    exclude_deterministic=False,
)
with SDESimulator(n_simulations=50, source="em_scan"):
    with Filter(
        filter_config=ContinuousTimeEnKFConfig(
            n_particles=50,
            record_filtered_states_mean=True,
            record_filtered_states_cov_diag=True,
        )
    ):
        samples_interp = predictive_interp(
            jr.PRNGKey(42),
            obs_times=obs_times_sparse,
            obs_values=obs_values_sparse,
            predict_times=predict_times_dense,
        )

pred_states_interp = jnp.asarray(samples_interp["f_predicted_states"])  # (num_samples, n_sim, T_dense, 3)
pred_times_interp = jnp.asarray(samples_interp["f_predicted_times"])  # (num_samples, n_sim, T_dense)

print("interp shapes:", pred_states_interp.shape, pred_times_interp.shape)
interp_draws = flatten_draws(pred_states_interp)
pred_times_interp_1d = flatten_draws(pred_times_interp)[0]

# Plot: sparse observations (scatter) vs dense posterior-predictive interpolation (bands)
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
lo = jnp.percentile(interp_draws, 2.5, axis=0)
hi = jnp.percentile(interp_draws, 97.5, axis=0)
state_labels = [r"x1", r"x2", r"x3"]
for i, ax in enumerate(axes):
    ax.fill_between(
        pred_times_interp_1d,
        lo[:, i],
        hi[:, i],
        alpha=0.3,
        label="95% CI (interpolation)",
    )
    ax.plot(
        times_backtest,
        states_backtest[:, i],
        "k--",
        label="True state",
        lw=1,
    )
    if i == 0:
        ax.scatter(
            obs_times_sparse,
            obs_values_sparse[:, 0],
            color="C3",
            marker="x",
            s=40,
            zorder=5,
            label=f"Observed (every {downsample}th)",
        )
    ax.set_ylabel(state_labels[i])
    ax.legend(loc="upper right", fontsize=8)
axes[0].set_title(
    "Back-tested real-time forecasting: sparse obs, dense predict_times"
)
axes[-1].set_xlabel("time")
plt.tight_layout()
plt.show()

Now we move to Deterministic continuous-time dynamical systems (ODEs)

First set up the model without diffusion

In [ ]:
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist

import dynestyx as dsx
from dynestyx import (
    ContinuousTimeStateEvolution,
    DynamicalModel,
    LinearGaussianObservation,
    flatten_draws,
)

state_dim = 3
observation_dim = 1


def continuous_time_deterministic_l63_model(rho=None, obs_times=None, obs_values=None, predict_times=None):
    """Model that samples drift parameter rho and uses it in dynamics (ODE, no diffusion)."""
    rho = numpyro.sample("rho", dist.Uniform(10.0, 40.0), obs=rho)

    # Create the dynamical model with sampled rho
    dynamics = DynamicalModel(
        initial_condition=dist.MultivariateNormal(
            loc=jnp.zeros(state_dim), covariance_matrix=2.0**2 * jnp.eye(state_dim)
        ),
        state_evolution=ContinuousTimeStateEvolution(
            drift=lambda x, u, t: jnp.array(
                [
                    10.0 * (x[1] - x[0]),
                    x[0] * (rho - x[2]) - x[1],
                    x[0] * x[1] - (8.0 / 3.0) * x[2],
                ]
            )
        ),
        observation_model=LinearGaussianObservation(
            H=jnp.eye(observation_dim, state_dim),
            R=jnp.eye(observation_dim),
        ),
    )

    return dsx.sample("f", dynamics, obs_times=obs_times, obs_values=obs_values, predict_times=predict_times)

Generate Data

In [ ]:
import jax.random as jr
from numpyro.infer import Predictive

from dynestyx import ODESimulator

# Generate longer trajectory: training portion + held-out future for rollout evaluation
n_train = 5.0
T_forecast = 2.0
obs_times_full = jnp.arange(0.0, n_train + T_forecast, 0.0025)

# Train / test split
obs_times = obs_times_full[obs_times_full <= n_train]
obs_times_test = obs_times_full[obs_times_full > n_train]

prng_key = jr.PRNGKey(0)
predictive_key = jr.split(prng_key, 2)[1]

predictive_model = Predictive(continuous_time_deterministic_l63_model, num_samples=1)

with ODESimulator():
    synthetic_samples = predictive_model(predictive_key, rho=28.0, predict_times=obs_times_full)

Visualisation of the data generated

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))

print(
    "synthetic shapes:",
    synthetic_samples["f_times"].shape,
    synthetic_samples["f_states"].shape,
    synthetic_samples["f_observations"].shape,
)
# Expected: (num_samples, n_sim, T, ...) with num_samples=n_sim=1 in this cell.
times = synthetic_samples["f_times"][0, 0, :]
states = synthetic_samples["f_states"][0, 0, :, :]
observations = synthetic_samples["f_observations"][0, 0, :, 0]

# Training portion for MCMC; future withheld for rollout eval
mask_train = times <= n_train
times_train = times[mask_train]
observations_train = observations[mask_train]
state_colors = sns.color_palette("tab10", 3)

for d in range(3):
    plt.plot(times, states[:, d], label=fr"xd+1", color=state_colors[d])

plt.scatter(
    times,
    observations,
    label=r"y1",
    marker="x",
    color="black",
    alpha=0.5,
)
plt.title("Synthetic Data: States and Observations")
plt.xlabel("Time")
plt.ylabel("Value")
sns.despine()
plt.legend()
plt.show()

Inferring parameter and starting hidden state with ODE Simulator and NUTS

In [ ]:
import jax.random as jr
from numpyro.infer import MCMC, NUTS

mcmc_key = jr.PRNGKey(42)

with ODESimulator():
    nuts_kernel = NUTS(continuous_time_deterministic_l63_model)
    mcmc = MCMC(nuts_kernel, num_samples=500, num_warmup=500)
    mcmc.run(mcmc_key, obs_times=times_train, obs_values=observations_train)

posterior_samples = mcmc.get_samples()

#Plot the inferred posterior distribution of rho
import arviz as az

az.style.use("arviz-white")

# For plotting reasons
parameter_posterior_samples = {"rho": posterior_samples["rho"]}

az.plot_posterior(
    parameter_posterior_samples, var_names=["rho"], hdi_prob=0.95, ref_val=28.0
)

plt.show()

Produce and plot smoothed estimates of hidden states

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))

times = synthetic_samples["f_times"][0, 0, :]
synthetic_states = synthetic_samples["f_states"][0, 0, :, :]
#Might not be a JAX array so we convert
states_raw = jnp.asarray(posterior_samples["f_states"])
# Unlist draw axes explicitly: (num_draws, T, D)
#Collect all samples generated by differnet chains all into one
states = states_raw.reshape((-1, states_raw.shape[-2], states_raw.shape[-1]))
print("posterior f_states shape:", states_raw.shape, "->", states.shape)

# Posterior states are at times_train (training portion only)
median = jnp.median(states, axis=0)  # (T, D)
p10 = jnp.percentile(states, 10, axis=0)  # (T, D)
p90 = jnp.percentile(states, 90, axis=0)  # (T, D)
state_colors = sns.color_palette("tab10", 3)

for d in range(3):
    plt.plot(
        times_train,
        synthetic_states[mask_train, d],
        label=fr"True xd+1",
        color=state_colors[d],
    )
    plt.fill_between(times_train, p10[:, d], p90[:, d], alpha=0.3, color=state_colors[d])
    plt.plot(
        times_train,
        median[:, d],
        label=fr"Posterior median xd+1",
        lw=2,
        ls="--",
        color=state_colors[d],
    )

plt.title("State Trajectories: Ground Truth and Posterior")
plt.xlabel("Time")
plt.ylabel("Value")
sns.despine()
plt.legend(ncol=2)
plt.show()

NameError: name 'synthetic_samples' is not defined

<Figure size 1000x500 with 0 Axes>

In [ ]:
#